# M03 lab solution: model demo

**Brief.** Wrap an open embedding or text model from the Hub in a Gradio app that shows its output for a user-entered query.

The model is `sentence-transformers/all-MiniLM-L6-v2`. It embeds the Larkspur handbook passages once at start-up; the app embeds a typed query and returns the three closest passages. The search function is also called directly on three golden questions, so the demo is checked without a browser.

No API key is needed. The first run downloads the model from the Hub.

In [ ]:
import sys
from pathlib import Path

from dotenv import load_dotenv

sys.path.insert(0, str(Path("../../../../src").resolve()))
load_dotenv()

DATA = Path("../data")

import gradio as gr
import numpy as np
from sentence_transformers import SentenceTransformer

In [ ]:
from lab_kit import is_hit, load_handbook, load_jsonl
from langchain_text_splitters import RecursiveCharacterTextSplitter

pages = load_handbook(DATA / "larkspur" / "handbook")
golden = load_jsonl(DATA / "larkspur" / "golden.jsonl")
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = [c for p in pages for c in splitter.split_text(p["text"])]
print(len(pages), "handbook files,", len(chunks), "chunks,", len(golden), "golden questions")

## 1. Load the model once and embed the passages

Loading inside the search function would reload the model on every query. `normalize_embeddings=True` makes the dot product a cosine similarity.

In [ ]:
MODEL_ID = "sentence-transformers/all-MiniLM-L6-v2"
model = SentenceTransformer(MODEL_ID)
passage_vectors = model.encode(chunks, normalize_embeddings=True, convert_to_numpy=True)
print(MODEL_ID, passage_vectors.shape, passage_vectors.dtype)

In [ ]:
def search(query: str, k: int = 3) -> list[tuple[float, str]]:
    """Top-k handbook passages for a query, as (cosine similarity, passage)."""
    if not query or not query.strip():
        raise gr.Error("Type a question first.")
    q = model.encode([query], normalize_embeddings=True, convert_to_numpy=True)[0]
    scores = passage_vectors @ q
    top = np.argsort(-scores)[:k]
    return [(float(scores[i]), chunks[i]) for i in top]

def search_markdown(query: str) -> str:
    return "\n\n---\n\n".join(f"**{score:.3f}**\n\n{text}" for score, text in search(query))

## 2. Call the search function directly on three golden questions

A passage is a hit when it contains one of the question's evidence spans (`lab_kit.is_hit`).

In [ ]:
import pandas as pd

check = []
for g in golden[:3]:
    hits = search(g["question"])
    check.append({"id": g["id"], "question": g["question"],
                  "top-3 scores": [round(s, 3) for s, _ in hits],
                  "hit ranks": [i for i, (_, t) in enumerate(hits, 1) if is_hit(t, g["evidence_spans"])]})
pd.DataFrame(check)

## 3. The Gradio app

`gr.Blocks` with a textbox, a button and a Markdown output. `prevent_thread_lock=True` returns control to the notebook, and `demo.close()` stops the server so nothing is left running.

In [ ]:
with gr.Blocks(title="Larkspur handbook search") as demo:
    gr.Markdown(f"# Larkspur handbook search\nModel: `{MODEL_ID}`. Returns the three closest passages.")
    query = gr.Textbox(label="Question", placeholder="How long does a refund take?")
    button = gr.Button("Search")
    output = gr.Markdown()
    button.click(search_markdown, inputs=query, outputs=output)
    query.submit(search_markdown, inputs=query, outputs=output)

demo.launch(prevent_thread_lock=True)

In [ ]:
demo.close()

## Results

In [ ]:
pd.DataFrame([{"id": c["id"], "hit in top 3": bool(c["hit ranks"]),
               "first hit rank": (c["hit ranks"] or [None])[0]} for c in check])

## What this shows

- The model is loaded once and every query reuses it, in the notebook and in the app alike.
- Passages are split at 500 characters, well inside the model's input limit; text past the limit would be dropped without an error.
- A Gradio demo has no login, rate limit or monitoring. It lets people try the model; production traffic needs a service with those controls.